# 02. Privacy-Enhancing Technologies (PETs)

## 📚 Learning Objectives

By completing this notebook, you will:
- Use privacy-enhancing technologies (PETs)
- Compare techniques and their trade-offs
- Choose appropriate PETs for use cases

## 🔗 Where this fits

**Builds on:** Course 05 (AIAT 115) — Unit 2, lesson 04 "Feature Transformation: Scaling and Encoding" — transformations that preserve utility; a PET is a transformation that also destroys identity.

---


# 02. Privacy-Enhancing Technologies (PETs)

## 🚨 THE PROBLEM: We Need to Compute on Encrypted Data

**Remember the limitation from the previous notebook?**

We learned basic data protection strategies like encryption, anonymization, and pseudonymization. But we discovered:

**What if we need to compute on encrypted data without decrypting it?**

**The Problem**: Advanced AI use cases often require:
- ❌ **Computing on encrypted data** without decryption
- ❌ **Collaborative computation** without sharing raw data
- ❌ **Privacy-preserving machine learning** on sensitive data
- ❌ **Advanced privacy guarantees** beyond basic protection

**We've learned:**
- ✅ How to encrypt sensitive data (Notebook 1)
- ✅ How to anonymize and pseudonymize data
- ✅ Basic data protection strategies

**But we haven't learned:**
- ❌ How to **compute on encrypted data** without decrypting
- ❌ How to enable **secure multi-party computation**
- ❌ How to use **homomorphic encryption** for privacy-preserving ML
- ❌ How to apply **advanced privacy technologies**

**We need privacy-enhancing technologies (PETs)** to:
1. Compute on encrypted data (homomorphic encryption)
2. Enable secure multi-party computation
3. Provide stronger privacy guarantees
4. Support privacy-preserving machine learning

**This notebook solves that problem** by teaching you advanced privacy-enhancing technologies like homomorphic encryption and secure multi-party computation!

---

## 📚 Prerequisites (What You Need First)

**BEFORE starting this notebook**, you should have completed:
- ✅ **Example 1: Data Protection** - Understanding basic protection strategies
- ✅ **Basic Python knowledge**: Functions, data manipulation
- ✅ **Understanding of encryption**: Basic encryption concepts (from Example 1)

**If you haven't completed these**, you might struggle with:
- Understanding why advanced privacy technologies are needed
- Knowing how homomorphic encryption works
- Understanding secure multi-party computation concepts

---

## 🔗 Where This Notebook Fits

**This is the SECOND example in Unit 3** - it teaches you advanced privacy technologies!

**Why this example SECOND?**
- **Before** you can use advanced PETs, you need basic data protection (Example 1)
- **Before** you can implement differential privacy, you need to understand PETs
- **Before** you can ensure GDPR compliance, you need privacy technologies

**Builds on**: 
- 📓 Example 1: Data Protection (we learned basic protection, now we learn advanced!)

**Leads to**: 
- 📓 Example 3: Differential Privacy (mathematical privacy guarantees)
- 📓 Example 4: GDPR Compliance (regulatory compliance)
- 📓 Example 5: Secure Development (secure coding practices)

**Why this order?**
1. PETs provide **advanced solutions** (needed after basic protection)
2. PETs enable **privacy-preserving ML** (critical for AI)
3. PETs show **cutting-edge techniques** (homomorphic encryption, SMPC)

---

## The Story: Computing Without Revealing

Imagine you're a bank that needs to calculate average account balances across multiple banks without revealing individual balances. **Before** advanced PETs, you'd have to share data (privacy risk!). **After** using secure multi-party computation, you can compute the average without any bank seeing others' data!

Same with AI: **Before** we encrypt data but can't compute on it, now we learn homomorphic encryption - compute on encrypted data without decrypting! **After** PETs, we can train models on encrypted data while preserving privacy!

---

## Why Privacy-Enhancing Technologies Matter

Privacy-enhancing technologies are essential for ethical AI:
- **Privacy-Preserving ML**: Train models on encrypted data
- **Collaborative AI**: Enable multi-party computation without data sharing
- **Strong Guarantees**: Provide mathematical privacy guarantees
- **Compliance**: Meet strict privacy regulations
- **Trust**: Build user confidence in privacy-preserving systems

## Learning Objectives
1. Understand homomorphic encryption concepts
2. Learn secure multi-party computation (SMPC)
3. Understand privacy-utility trade-offs
4. Compare different PETs
5. Apply PETs to privacy-preserving machine learning
6. Understand when to use each technology

## 📌 The case: the keyboard on a billion phones trains a model it never uploads

Federated learning is not a thought experiment. **Google deployed it in Gboard**, the
Android keyboard, from **2017** onward: the phone stores the local context and whether you
accepted a suggestion, trains on-device, and sends **model updates rather than keystrokes**
to the server, which averages them (McMahan et al., 2017 — *Communication-Efficient Learning
of Deep Networks from Decentralized Data*, the FedAvg paper this notebook implements).
Google has since reported using the same machinery for next-word prediction, emoji
suggestion and out-of-vocabulary word discovery.

Take seriously what that means. Your typing — the single most sensitive text stream you
produce — improves a shared model that no engineer at Google can read. That is a real
privacy-enhancing technology in production on hundreds of millions of devices, not a
research demo.

**And it is not a privacy guarantee.** In **Deep Leakage from Gradients** (Zhu, Liu & Han,
NeurIPS 2019) researchers recovered private training data *from the shared gradients alone*,
**pixel-wise accurate for images and token-wise matching for text**. Parameters are not
inert; they are a lossy encoding of the data that produced them. That is exactly why
production federated systems combine FL with **differential privacy** — the subject of the
next notebook — rather than relying on FL by itself.

**The WHY — what goes wrong without PETs.** The default alternative is to pool the data: three
hospitals ship patient records to one server so a model can be trained. That single server is
now a target worth attacking, a jurisdiction problem, a consent problem and a permanent
liability. PETs exist so that the useful computation can happen **without the pooled copy
ever existing**. You are about to price that: three hospitals will compute a joint total
without revealing their counts, and a federated model on real diagnostic data will be scored
against the centralised model trained on the pooled version.

---


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- **`sklearn.datasets.load_breast_cancer`** - a real diagnostic dataset: 569 real
  tumour biopsies from the Wisconsin Diagnostic Breast Cancer study, with 30 real
  measured features and a real malignant/benign label. Medical records are exactly
  the kind of data these privacy technologies exist to protect, so we use real ones.
- Textbook RSA with a deliberately tiny key, for the homomorphic-property demo only.

**Outputs:** What you'll see when you run the cells

- Arithmetic performed on ciphertexts and checked against the plaintext answer
- Three "hospitals" computing a real joint total without revealing their real counts
- A federated model trained on real patient data that never leaves its hospital,
  compared against a centralised model trained on the pooled data

---

## Part 1: Homomorphic Encryption - Computing on Encrypted Data

**Idea**: some encryption schemes let you do arithmetic **directly on ciphertexts**;
decrypting the result gives the answer you would have computed on the plaintexts.

We demonstrate the idea with **textbook RSA**, which is *multiplicatively homomorphic*:
`E(a) x E(b) = E(a x b)`. ⚠️ The tiny key below is deliberately insecure - it is a
teaching model of the *property*, not production crypto. Real systems use schemes like
Paillier (additive) or BFV/CKKS ("fully homomorphic"), via libraries such as SEAL or
OpenFHE.

In [1]:
# Why: the homomorphic property lets an untrusted server COMPUTE on data
# it can never read - the core promise behind privacy-preserving cloud AI.

# Step 1: Demonstrate the homomorphic property with textbook RSA (teaching only!)

print("="*80)
print("🔐 HOMOMORPHIC ENCRYPTION: COMPUTE ON ENCRYPTED DATA")
print("="*80)

# Tiny textbook RSA keypair (INSECURE - for demonstration only)
p, q = 61, 53
n = p * q                      # modulus (public)
phi = (p - 1) * (q - 1)
e = 17                         # public exponent
d = pow(e, -1, phi)            # private exponent

def rsa_encrypt(m):
    return pow(m, e, n)

def rsa_decrypt(ct):
    return pow(ct, d, n)

a, b = 7, 6
ct_a, ct_b = rsa_encrypt(a), rsa_encrypt(b)
print(f"\nPlaintexts:        a = {a}, b = {b}")
print(f"Ciphertexts:       E(a) = {ct_a}, E(b) = {ct_b}")

# Multiply the CIPHERTEXTS - without ever decrypting the inputs
ct_product = (ct_a * ct_b) % n
result = rsa_decrypt(ct_product)
print(f"E(a) * E(b) mod n = {ct_product}   (still encrypted)")
print(f"Decrypt(E(a)*E(b)) = {result}   <- equals a*b = {a*b}!")
assert result == a * b
print("\n✅ We computed a*b while the server only ever saw ciphertexts.")
print("   (Real HE libraries support addition too, and much larger data.)")

🔐 HOMOMORPHIC ENCRYPTION: COMPUTE ON ENCRYPTED DATA

Plaintexts:        a = 7, b = 6
Ciphertexts:       E(a) = 2369, E(b) = 824
E(a) * E(b) mod n = 2557   (still encrypted)
Decrypt(E(a)*E(b)) = 42   <- equals a*b = 42!

✅ We computed a*b while the server only ever saw ciphertexts.
   (Real HE libraries support addition too, and much larger data.)


## Part 2: Secure Multi-Party Computation (SMPC)

**Idea**: several parties compute a joint result (say, a total or average) without any
party revealing its private input. The classic building block is **additive secret
sharing**: each party splits its number into random-looking shares that sum to the true
value, and only the *sum of everything* is ever reconstructed.

In [2]:
# Step 2: Additive secret sharing - three hospitals compute a total
# number of malignant cases without revealing their individual counts

import random
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

random.seed(42)

print("="*80)
print("🤝 SECURE MULTI-PARTY COMPUTATION (additive secret sharing)")
print("="*80)

# Load REAL diagnostic records: 569 real biopsies from the Wisconsin Diagnostic
# Breast Cancer study. In sklearn's encoding, target 0 = malignant, 1 = benign.
cancer = load_breast_cancer()
X_all, y_all = cancer.data, cancer.target
is_malignant = (y_all == 0).astype(int)
print(f"\nReal diagnostic records loaded: {len(y_all)}")
print(f"Real malignant cases in the full dataset: {is_malignant.sum()} "
      f"({is_malignant.mean():.1%})")

# Split the real records across three "hospitals". Each hospital's malignant
# count is now a REAL number derived from real records - and is exactly the
# kind of figure a hospital would refuse to publish on its own.
splits = np.array_split(np.arange(len(y_all)), 3)
hospital_names = ['Hospital A', 'Hospital B', 'Hospital C']
hospital_counts = {name: int(is_malignant[idx].sum())
                   for name, idx in zip(hospital_names, splits)}
hospital_sizes = {name: len(idx) for name, idx in zip(hospital_names, splits)}

print("\nEach hospital's PRIVATE count (what it will never reveal):")
for name in hospital_names:
    print(f"  {name}: {hospital_counts[name]} malignant "
          f"out of {hospital_sizes[name]} patients")

n_parties = len(hospital_counts)
MOD = 10**9 + 7   # arithmetic is done modulo a large number

def make_shares(secret, n):
    """Split secret into n random shares that sum to secret (mod MOD).

    The randomness here IS the privacy mechanism: each individual share is
    uniformly random and therefore carries no information about the secret.
    """
    shares = [random.randrange(MOD) for _ in range(n - 1)]
    shares.append((secret - sum(shares)) % MOD)
    return shares

# Each hospital splits its private count into 3 shares
all_shares = {name: make_shares(v, n_parties) for name, v in hospital_counts.items()}
print("\nEach hospital's shares (look like random numbers - reveal nothing alone):")
for name, shares in all_shares.items():
    print(f"  {name}: {shares}")

# Share j of every hospital goes to party j; each party sums what it received
partial_sums = [sum(all_shares[name][j] for name in hospital_counts) % MOD
                for j in range(n_parties)]
print(f"\nEach party's partial sum: {partial_sums}")

total = sum(partial_sums) % MOD
true_total = sum(hospital_counts.values())
print(f"\nReconstructed TOTAL malignant cases: {total}")
print(f"True total (for checking):           {true_total}")
assert total == true_total
print(f"\n✅ The three hospitals learned that {total} of their "
      f"{sum(hospital_sizes.values())} combined")
print("   patients had malignant tumours - a genuinely useful public-health")
print("   figure - and no hospital ever saw another's raw count.")

🤝 SECURE MULTI-PARTY COMPUTATION (additive secret sharing)

Real diagnostic records loaded: 569
Real malignant cases in the full dataset: 212 (37.3%)

Each hospital's PRIVATE count (what it will never reveal):
  Hospital A: 97 malignant out of 190 patients
  Hospital B: 72 malignant out of 190 patients
  Hospital C: 43 malignant out of 189 patients

Each hospital's shares (look like random numbers - reveal nothing alone):
  Hospital A: [686579303, 119540831, 193879970]
  Hospital B: [26855092, 796233790, 176911197]
  Hospital C: [295310485, 262950628, 441738937]

Each party's partial sum: [8744873, 178725242, 812530104]

Reconstructed TOTAL malignant cases: 212
True total (for checking):           212

✅ The three hospitals learned that 212 of their 569 combined
   patients had malignant tumours - a genuinely useful public-health
   figure - and no hospital ever saw another's raw count.


## Part 3: Federated Learning - a PET for Model Training

**Federated learning (FL)** applies the same "don't share the raw data" principle to
*machine-learning training*:

1. Each client (hospital, phone, bank) keeps its data **locally**
2. Each client trains a model on its own data
3. Only the **model parameters** (weights) are sent to a server
4. The server **averages** the parameters into a global model (FedAvg)

FL is a privacy-enhancing technology because raw data never leaves the client - only
parameter updates travel. (Parameters can still leak information, which is why FL is often
combined with differential privacy - the topic of the next notebook.)

> You will build exactly this simulation yourself in **Exercise 2 (Task 2)** of this unit.

In [3]:
# Why: federated learning trains one shared model while every client's raw
# data stays on the client - only model parameters travel.

# Step 3: A federated learning simulation (FedAvg on real patient records)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

print("="*80)
print("🌐 FEDERATED LEARNING SIMULATION (real diagnostic data)")
print("="*80)

# Same REAL dataset as Step 2: 569 real biopsies, 30 real measured features.
# The task is the real clinical one - predict malignant vs benign.
X, y = cancer.data, cancer.target
print(f"\nReal records: {X.shape[0]} patients, {X.shape[1]} measured features")
print(f"Real class balance: {(y==0).sum()} malignant / {(y==1).sum()} benign")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
# Scale on the training set only - these features have wildly different units
# (cell area in the hundreds, smoothness around 0.1).
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

# Split the training data across 3 hospitals (data stays "local")
n_clients = 3
client_X = np.array_split(X_train_s, n_clients)
client_y = np.array_split(y_train, n_clients)

# Each client trains locally; only coefficients are "sent" to the server
coefs, intercepts = [], []
for i in range(n_clients):
    local = LogisticRegression(max_iter=1000).fit(client_X[i], client_y[i])
    coefs.append(local.coef_)
    intercepts.append(local.intercept_)
    print(f"{hospital_names[i]}: trained on {len(client_y[i])} local patient records "
          f"({(client_y[i]==0).sum()} malignant), "
          f"local accuracy {local.score(client_X[i], client_y[i]):.3f}")

# Server: average the parameters (FedAvg)
global_model = LogisticRegression(max_iter=1000).fit(X_train_s[:10], y_train[:10])  # init shell
global_model.coef_ = np.mean(coefs, axis=0)
global_model.intercept_ = np.mean(intercepts, axis=0)

# Compare with a centralized model (which needed ALL the raw records in one place)
central = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
acc_fed = accuracy_score(y_test, global_model.predict(X_test_s))
acc_central = accuracy_score(y_test, central.predict(X_test_s))
print(f"\nFederated (averaged) model test accuracy: {acc_fed:.3f}")
print(f"Centralized model test accuracy:          {acc_central:.3f}")
print(f"Cost of keeping the data local:           {acc_fed - acc_central:+.3f}")

if acc_fed >= acc_central - 0.02:
    print("\n✅ Essentially the same clinical accuracy - but in FL, no patient")
    print("   record ever left the hospital that collected it.")
else:
    print(f"\n⚠️  Federated averaging cost {acc_central - acc_fed:.3f} accuracy here.")
    print("   That is a real trade-off, not a failure: this run used a single")
    print("   averaging round on non-identically-distributed local data, which is")
    print("   exactly when FedAvg is weakest. Real deployments run many rounds.")

print("\nNOTE on what FL does NOT protect: the parameters themselves can leak")
print("information about the training data (membership-inference and gradient-")
print("inversion attacks). That is why production FL is combined with differential")
print("privacy - the subject of the next notebook.")

🌐 FEDERATED LEARNING SIMULATION (real diagnostic data)

Real records: 569 patients, 30 measured features
Real class balance: 212 malignant / 357 benign
Hospital A: trained on 133 local patient records (48 malignant), local accuracy 0.992
Hospital B: trained on 133 local patient records (53 malignant), local accuracy 0.985
Hospital C: trained on 132 local patient records (47 malignant), local accuracy 1.000

Federated (averaged) model test accuracy: 0.982
Centralized model test accuracy:          0.988
Cost of keeping the data local:           -0.006

✅ Essentially the same clinical accuracy - but in FL, no patient
   record ever left the hospital that collected it.

NOTE on what FL does NOT protect: the parameters themselves can leak
information about the training data (membership-inference and gradient-
inversion attacks). That is why production FL is combined with differential
privacy - the subject of the next notebook.


In [4]:
# Why compare: each privacy-enhancing technology (PET) protects something
# different at a different cost - choosing the right one is a design decision.

# Step 4: When to use which PET?
import pandas as pd

# One row per PET: what it protects, where it is used, and what it costs.
pet_comparison = pd.DataFrame([
    ['Homomorphic encryption', 'Compute on encrypted data',
     'Outsourced/cloud computation', 'High computational cost'],
    ['Secure multi-party computation', 'Joint result, private inputs',
     'Cross-organization statistics', 'Communication overhead'],
    ['Federated learning', 'Train models without pooling data',
     'Mobile keyboards, hospitals', 'Updates can still leak info'],
    ['Differential privacy (next!)', 'Provable bound on individual leakage',
     'Published statistics, ML training', 'Noise costs accuracy'],
], columns=['Technology', 'What it protects', 'Typical use', 'Main cost'])

print(pet_comparison.to_string(index=False))

                    Technology                     What it protects                       Typical use                   Main cost
        Homomorphic encryption            Compute on encrypted data      Outsourced/cloud computation     High computational cost
Secure multi-party computation         Joint result, private inputs     Cross-organization statistics      Communication overhead
            Federated learning    Train models without pooling data       Mobile keyboards, hospitals Updates can still leak info
  Differential privacy (next!) Provable bound on individual leakage Published statistics, ML training        Noise costs accuracy


## 💬 Discuss

Your run measured the price: federated **0.982** vs centralised **0.988** test accuracy on
real breast-cancer diagnoses — **0.006** for keeping every record inside the hospital that
collected it.

1. **Is 0.6 points a bargain or a body count?** This is a malignancy classifier. Six
   thousandths of accuracy on a cancer-screening task is not an abstraction; on a large
   enough population it is a number of missed diagnoses. Argue the case *for* pooling the
   data — genuinely, in your own words — and then say what safeguard would have to be in
   place before you would sign it off. If your answer is "always federate", explain it to
   the patient who was missed.
2. **What exactly did the three hospitals learn?** Secret sharing revealed only the total
   (212). But if there are three parties and two of them collude, they can subtract their own
   inputs and recover the third's number exactly. Does "secure multi-party computation"
   remain an honest description of the protocol you ran? What minimum number of participants
   would you require before using it in a real consortium, and who verifies that they are
   genuinely independent organisations?
3. **Who is accountable when nobody holds the data?** In a federated system there is no
   central dataset, so there is also no single place to honour a GDPR erasure request, no
   single dataset to audit for bias (Unit 2), and no single owner to subpoena. Is federated
   learning a privacy improvement, an accountability regression, or both? Say which regulator
   you would expect to be least happy with it, and why.


---

## 🚫 When Privacy Technologies Hit a Limitation

### The Limitation We Discovered

We've learned advanced privacy-enhancing technologies like homomorphic encryption and secure multi-party computation. **But there's still a challenge:**

**How do we provide mathematical privacy guarantees?**

Privacy technologies work well when:
- ✅ We can use homomorphic encryption for specific operations
- ✅ We can enable secure multi-party computation
- ✅ We understand privacy-utility trade-offs

**But we need stronger guarantees:**
- ❌ **Mathematical privacy guarantees** (not just techniques)
- ❌ **Quantifiable privacy protection** (measurable privacy loss)
- ❌ **Formal privacy definitions** (differential privacy)
- ❌ **Provable privacy bounds** (epsilon-delta guarantees)

### Why This Is a Problem

When we use privacy technologies without formal guarantees:
- We don't know how much privacy we're actually providing
- We can't quantify privacy loss
- We can't prove our systems are private
- We may think we're private but aren't

### The Solution: Differential Privacy

We need **differential privacy** to:
1. Provide mathematical privacy guarantees
2. Quantify privacy loss (epsilon parameter)
3. Prove privacy protection formally
4. Enable privacy-preserving data analysis with provable guarantees

**This is exactly what we'll learn in the next notebook: Differential Privacy!**

---

## ➡️ Next Steps

**You've completed this notebook!** Now you understand:
- ✅ How to use basic data protection (Notebook 1)
- ✅ How to use advanced privacy technologies (This notebook!)
- ✅ **The limitation**: We need mathematical privacy guarantees!

**Next notebook**: `03_differential_privacy.ipynb`
- Learn about differential privacy and epsilon parameter
- Understand mathematical privacy guarantees
- Apply differential privacy to data analysis

## ⚠️ Where this breaks

- **Parameters leak.** Zhu et al. (2019) recovered training data from gradients, pixel-wise
  for images and token-wise for text. "The raw data never left the device" is a statement
  about transport, not about information. Treat model updates as sensitive data with a
  smaller constant in front, and combine FL with DP (Notebook 03) or secure aggregation when
  the stakes justify it.
- **The RSA cell in this notebook is a teaching model of a property, not cryptography.** The
  key is deliberately tiny and textbook RSA is deterministic and malleable — the very
  homomorphic property being demonstrated is what makes it unsafe as an encryption scheme.
  Production work uses Paillier, BFV or CKKS via SEAL or OpenFHE, and the arithmetic is
  orders of magnitude slower than the plaintext version you just ran.
- **Homomorphic encryption is priced per operation, and the price is steep.** Fully
  homomorphic schemes remain thousands of times slower than plaintext arithmetic and support
  a restricted set of operations with a noise budget that must be managed. It is the right
  tool for a small, well-defined computation on outsourced data — not for training a deep
  network.
- **Secret sharing assumes non-collusion and honest participation.** The scheme you ran gives
  perfect privacy against a passive, non-colluding adversary. It does not detect a party who
  submits a deliberately wrong share, and it collapses when all-but-one collude.
- **Federated averaging assumes clients that look roughly alike.** Our three hospitals were
  produced by splitting one dataset, so they are close to independent and identically
  distributed. Real hospitals are not: different populations, different equipment, different
  labelling practice. Under strong non-IID data, plain FedAvg can converge slowly or to a
  worse model than any single participant would have trained alone.
- **None of these technologies decides *whether* the computation should happen.** They make
  a chosen computation more private. Whether the model should exist at all is Unit 1's
  question, and no amount of encryption answers it.

**The cheaper alternative, and the first thing to check:** can the question be answered from
**aggregate statistics you are already allowed to publish**? A great deal of "we need to pool
the data" turns out to be "we need three counts", and three counts do not require a
cryptographic protocol — only a lawyer and an email.


## 📚 References

1. Rivest, R. L., Shamir, A. & Adleman, L. (1978). *A Method for Obtaining Digital Signatures and Public-Key Cryptosystems*. Communications of the ACM, 21(2).
2. Gentry, C. (2009). *Fully Homomorphic Encryption Using Ideal Lattices*. STOC 2009.
3. McMahan, H. B., Moore, E., Ramage, D., Hampson, S. & Agüera y Arcas, B. (2017). *Communication-Efficient Learning of Deep Networks from Decentralized Data*. AISTATS 2017. <https://arxiv.org/abs/1602.05629>
4. Kairouz, P., McMahan, H. B., et al. (2021). *Advances and Open Problems in Federated Learning*. Foundations and Trends in Machine Learning, 14(1-2). <https://arxiv.org/abs/1912.04977>